# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Get metadata as a JSON dict
metadata = dataset.metadata.to_json()
# Print dataset name and description
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All references are via `@id` fields for full traceability and reproducibility.

In [ ]:
# List available record sets, fields, and columns with their @id
record_sets_info = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets_info:
    print(f"Record set name: {rs.name}; @id: {rs['@id']}")
    print(" Fields:")
    for field in rs.fields:
        print(f"  - Field: {field.name}; @id: {field['@id']}; dataType: {field.data_type}")
        if hasattr(field, 'column') and field.column is not None:
            print(f"      Column @id: {field.column['@id']}")
    print()

Below, we print a sample of records from one record set, using the required `@id` for reference.
You can adjust the selected `record_set_id` based on the overview above.

<span style="font-size:13px">Choose a record set you want to explore:</span>

In [ ]:
# Choose the @id of the main record set for survey responses
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    if 'survey' in rs.name.lower() or 'main' in rs.name.lower():
        main_record_set_id = rs['@id']
        break
if main_record_set_id is None:
    # Take the first record set if not found
    main_record_set_id = dataset.metadata.record_sets[0]['@id']

print(f"Printing sample records from the record set: {main_record_set_id}")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    if i < 5:
        print(record)
    else:
        break

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print available columns in the main record set
print(f"Columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We use fields referenced by their `@id` for traceability.

**Example analysis:**
1. Filter by a numeric mental health score (e.g., GAD-7 score)
2. Normalize the score
3. Group by an attribute, such as participant's village or gender.

In [ ]:
main_df = dataframes[main_record_set_id]

# Identify numeric field @id (e.g., GAD-7)
gad7_field_id = None
group_field_id = None
for rs in dataset.metadata.record_sets:
    if rs['@id'] == main_record_set_id:
        for field in rs.fields:
            if 'gad7' in field.name.lower() or 'GAD-7' in field.name:
                if field.data_type in ['Float', 'Integer', 'Number']:
                    gad7_field_id = field['@id']
            if 'village' in field.name.lower():
                group_field_id = field['@id']

if gad7_field_id is None:
    # Guess a numeric field
    for col in main_df.columns:
        if ('gad' in col.lower() or 'score' in col.lower()) and pd.api.types.is_numeric_dtype(main_df[col]):
            gad7_field_id = col
            break

if group_field_id is None:
    for col in main_df.columns:
        if 'village' in col.lower():
            group_field_id = col

# Filter records by GAD-7 score
threshold = 10
if gad7_field_id in main_df.columns:
    filtered_df = main_df[main_df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the GAD-7 scores
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    display(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by village (or similar field)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        print(f"Average normalized scores grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Could not find a GAD-7 numeric field for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following example visualizes the distribution of GAD-7 scores and the average score per village, if available.

In [ ]:
# Plot distribution of GAD-7 scores
if gad7_field_id and gad7_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[gad7_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.title("Distribution of GAD-7 Scores")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    # Bar plot: average GAD-7 score per village
    if group_field_id and group_field_id in main_df.columns:
        village_means = main_df.groupby(group_field_id)[gad7_field_id].mean().sort_values()
        plt.figure(figsize=(10, 5))
        village_means.plot(kind='bar', color='orchid')
        plt.title("Average GAD-7 Score per Village")
        plt.ylabel("Average GAD-7 Score")
        plt.xlabel("Village")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric GAD-7 field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. By referencing entities solely by their `@id`, we demonstrated how to:
- Review available record sets, fields, and columns.
- Load and filter responses based on numeric scores (e.g., GAD-7).
- Normalize and group scores by demographic attributes (such as village).
- Visualize survey data distributions and community variations in mental health indicators.

This approach ensures reproducibility, traceability, and alignment with FAIR data principles. The dataset can support further research and community health interventions leveraging AI-ready standards.